In [41]:
from sbident import SBIdent
from astropy.time import Time
from astropy.coordinates import SkyCoord
import astropy.units as u
import lkspacecraft
import pandas as pd
import numpy as np
from astropy.coordinates import match_coordinates_sky

from astroquery.jplhorizons import Horizons

In [153]:
mpc_obs = 'C57'                         #observer location (MPC observatory code)
time = Time('2026-01-15 23:59:00')      #observing time
center = SkyCoord(129.8241886055643, 16.18114192546954, unit="deg")  #observing field of view center

In [154]:
ts = lkspacecraft.TESSSpacecraft()
tess_pos = ts.get_spacecraft_position(time=time, observer="EARTH BARYCENTER")
tess_vel = ts.get_spacecraft_velocity(time=time, observer="EARTH BARYCENTER")

In [155]:
tess_vec = pd.DataFrame(np.atleast_2d(tess_pos), columns=["x","y","z"])
tess_vec[["vx","vy","vz"]] = tess_vel
tess_vec

,x,y,z,vx,vy,vz
0,205458.636232,1121.27359,-201310.850722,0.108289,0.758488,0.739249


In [156]:
tess_vec = tess_vec.values[0]  # take the first row

# form the xobs dictionary that is the input for SBIdent location argument
xobs = ",".join([np.format_float_scientific(s, precision=5) for s in tess_vec])
xobs_location = {"xobs": xobs}

filters = None

In [157]:
xobs_location

{'xobs': '2.05459e+05,1.12127e+03,-2.01311e+05,1.08289e-01,7.58488e-01,7.39249e-01'}

In [158]:
sbid3 = SBIdent(xobs_location, time, center, maglim=21, hwidth=1)
sbid3_elem = SBIdent(xobs_location, time, center, maglim=21, hwidth=1, elem=True)

In [160]:
jpl_sb = sbid3.results.to_pandas()
jpl_sb_elem = sbid3_elem.results.to_pandas()
jpl_sb = pd.merge(jpl_sb, jpl_sb_elem, on="Object name")
jpl_sb

,Object name,Astrometric RA (hh:mm:ss),"Astrometric Dec (dd mm'ss"")","Dist. from center RA ("")","Dist. from center Dec ("")","Dist. from center Norm ("")",Visual magnitude (V),"RA rate (""/h)","Dec rate (""/h)",Absolute magntiude (H),Magnitude slope (G),Eccentricity,Perihelion (au),Time of perihelion passage (JD),Longitude of ascending node (deg),Argument of perihelion (deg),Inclination (deg),Epoch (JD)
0,3938 Chapront (1949 PL),08:43:00.39,"+16 15'42.9""",3.E3,291.,3219.,17.3,-3.123E+01,8.335E+00,13.90,0.15,0.038982729,2.3939200,2461475.10164,157.76924,75.056523,2.0894432,2461000.5
1,6049 Toda (1991 VP),08:39:40.48,"+15 22'06.5""",327.,-3.E3,2944.,17.6,-3.333E+01,8.129E+00,13.83,0.15,0.133534000,2.0742430,2460419.96990,174.74506,137.08617,2.4516620,2461000.5
2,6946 (1980 RX1),08:35:27.07,"+15 37'24.4""",-3.E3,-2.E3,3887.,16.5,-3.507E+01,1.263E+01,13.76,0.15,0.139220885,1.9525399,2460792.44400,143.81956,246.06250,4.5633023,2461000.5
3,7372 Emimar (1979 HH),08:40:30.96,"+17 07'08.8""",1.E3,3.E3,3537.,17.0,-2.853E+01,8.320E+00,12.85,0.15,0.032689049,2.7563011,2460145.56290,140.92230,156.67138,2.7827042,2461000.5
4,8466 Leyrat (1981 EV34),08:42:10.59,"+16 16'16.9""",2.E3,325.,2510.,19.2,-3.152E+01,8.413E+00,14.80,0.15,0.221663629,2.0126349,2460498.49820,156.71449,179.73978,2.4606809,2461000.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,823298 (2016 CV166),08:39:43.17,"+16 30'13.2""",365.,1.E3,1217.,20.9,-2.799E+01,4.655E+00,17.48,0.15,0.150716991,2.4861967,2461161.25539,245.90997,265.20522,1.3719663,2461000.5
161,870309 (2016 XA32),08:39:03.55,"+15 57'37.5""",-205.,-795.,820.7,20.7,-3.515E+01,-8.551E+00,17.59,0.15,0.113605941,2.3854018,2461015.36093,296.23714,175.18430,12.395884,2461000.5
162,(2001 RO157),08:35:11.34,"+17 09'23.0""",-4.E3,4.E3,4987.,20.5,-3.019E+01,9.858E+00,18.32,0.15,0.198922493,1.9878126,2460964.97648,140.51707,306.29326,2.4545789,2461000.5
163,(2012 DP113),08:40:16.68,"+15 31'09.4""",848.,-2.E3,2530.,21.0,-3.392E+01,4.254E-01,18.45,0.15,0.204507736,1.8585092,2461264.47910,278.86174,283.40844,3.8456719,2461000.5


In [161]:
jpl_sb["Astrometric Dec (dd mm'ss\")"] = [
            x.replace(" ", ":").replace("'", ":").replace('"', "")
            for x in jpl_sb["Astrometric Dec (dd mm'ss\")"]
        ]

coord = SkyCoord(
    jpl_sb[
        ["Astrometric RA (hh:mm:ss)", "Astrometric Dec (dd mm'ss\")"]
    ].values,
    frame="icrs",
    unit=(u.hourangle, u.deg),
)
jpl_sb["ra"] = coord.ra.deg
jpl_sb["dec"] = coord.dec.deg

In [162]:
catalog = SkyCoord(ra=jpl_sb["ra"].values * u.degree, 
                   dec=jpl_sb["dec"].values * u.degree)
catalog

<SkyCoord (ICRS): (ra, dec) in deg
    [(130.751625  , 16.26191667), (129.91866667, 15.36847222),
     (128.86279167, 15.62344444), (130.129     , 17.11911111),
     (130.544125  , 16.27136111), (129.458125  , 15.80680556),
     (129.25125   , 16.85497222), (130.27845833, 16.89977778),
     (130.43066667, 16.79297222), (129.45658333, 17.00952778),
     (130.86004167, 15.92802778), (130.234125  , 17.05527778),
     (129.14608333, 15.26227778), (130.53279167, 16.03286111),
     (130.118     , 15.78311111), (129.659125  , 15.93244444),
     (130.25383333, 16.23955556), (130.15575   , 17.12877778),
     (129.42633333, 15.46347222), (128.96645833, 16.57647222),
     (130.666     , 15.92691667), (128.796625  , 16.71855556),
     (129.26416667, 16.26033333), (129.74858333, 16.42505556),
     (130.558875  , 15.44922222), (130.23275   , 16.56036111),
     (128.85433333, 15.93330556), (129.113625  , 16.4605    ),
     (129.21954167, 15.84233333), (130.3245    , 17.12936111),
     (129.20383333, 

In [146]:
center = SkyCoord([129.8241886055643], [16.18114192546954], unit="deg")

In [147]:
idxc, idxcatalog, d2d, d3d = catalog.search_around_sky(center, 1 * u.deg)
np.all(d2d < 1*u.deg)

True

In [148]:
d2d.to("deg").value

array([0.89416904, 0.81773974, 0.98237776, 0.51376667, 0.84411085,
       0.95833842, 0.69674773, 0.29497798, 0.41667815, 0.99945656,
       0.81336886, 0.84798581, 0.25447899, 0.5454067 , 0.96440436,
       0.85909931, 0.78989548, 0.85084023, 0.18296803, 0.34906835,
       0.744296  , 0.95506794])

In [149]:
jpl_sb["Distnace to 3I (deg)"] = np.nan

In [150]:
jpl_sb["Distnace to 3I (deg)"].iloc[idxcatalog] = d2d.to("deg").value

You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [151]:
print(jpl_sb.iloc[idxcatalog[np.argsort(d2d)]].loc[:, ["Object name", "Visual magnitude (V)", "Distnace to 3I (deg)", "Perihelion (au)"]])

                      Object name Visual magnitude (V)  Distnace to 3I (deg)  \
21              78457 (2002 RZ31)                 18.7              0.182968   
15              46996 (1998 TR28)                 18.6              0.254479   
9                27746 (1990 WE3)                 18.8              0.294978   
22              99374 (2001 YH11)                 18.7              0.349068   
10               27812 (1993 OJ8)                 18.5              0.416678   
4   9395 Saint Michel (1994 PC39)                 17.3              0.513767   
16                48501 (1993 FM)                 17.4              0.545407   
8               27143 (1998 XK63)                 18.8              0.696748   
23             114441 (2003 AK14)                 18.0              0.744296   
19             70297 (1999 RG129)                 18.7              0.789895   
12              33984 (2000 NU24)                 18.0              0.813369   
1             6049 Toda (1991 VP)       